# Самостоятельное задание №1
## Применение CNN для классификации подлинных и поддельных изображений

В этом ноутбуке собран отчёт по теме применения свёрточной нейронной сети для задачи бинарной классификации изображений: "подлинное" против "поддельное".

Акценты отчёта:
- обоснование выбора архитектуры CNN;
- демонстрация transfer learning и fine-tuning;
- оценка качества на метриках и матрице ошибок;
- интерпретация ошибок и визуализация Grad-CAM.

## 1. Постановка задачи

Цель работы - построить модель, которая по изображению определяет, является ли оно подлинным или содержит признаки подделки, монтажа или фальсификации. Для таких задач важно не только распознать крупные формы, но и уловить мелкие текстурные артефакты, следы сжатия, неоднородности печати и аномалии в локальных областях изображения.

Для этого разумно использовать предобученную CNN с transfer learning, а затем выполнить тонкую настройку части верхних слоёв под специфику датасета.

In [ ]:
!pip -q install kagglehub

import copy
import random
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, models, transforms

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['font.size'] = 12

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
# Датасет скачивается в Colab через kagglehub
DATASET_ROOT = Path(kagglehub.dataset_download('robinreni/signature-verification-dataset'))
print('Path to dataset files:', DATASET_ROOT)

## 2. Подготовка данных

Ниже используется стандартная структура `ImageFolder`. Для бинарной задачи удобно хранить данные так:

data/
- train/
  - real/
  - fake/
- val/
  - real/
  - fake/
- test/
        "class TransformedSubset(Dataset):\n",
        "    def __init__(self, dataset, indices, transform):\n",
        "        self.dataset = dataset\n",
        "        self.indices = list(indices)\n",
        "        self.transform = transform\n",
        "        self.classes = dataset.classes\n",
        "\n",
        "    def __len__(self):\n",
        "        return len(self.indices)\n",
        "\n",
        "    def __getitem__(self, index):\n",
        "        image, label = self.dataset[self.indices[index]]\n",
        "        image = self.transform(image)\n",
        "        return image, label\n",
        "\n",
  - real/
        "    base_train_dataset = datasets.ImageFolder(root=str(train_dir), transform=None)\n",

Если отдельной валидационной выборки нет, ноутбук автоматически разобьёт обучающую часть на train/val.
        "        train_dataset = datasets.ImageFolder(root=str(train_dir), transform=train_transform)\n",
        "        val_dataset = datasets.ImageFolder(root=str(val_dir), transform=eval_transform)\n",
<VSCode.Cell id="#VSC-27567b8b" language="python">
        "        val_size = max(1, int(0.2 * len(base_train_dataset)))\n",
        "        train_size = len(base_train_dataset) - val_size\n",
        "        permutation = torch.randperm(len(base_train_dataset), generator=torch.Generator().manual_seed(42)).tolist()\n",
        "        train_indices = permutation[:train_size]\n",
        "        val_indices = permutation[train_size:]\n",
        "        train_dataset = TransformedSubset(base_train_dataset, train_indices, train_transform)\n",
        "        val_dataset = TransformedSubset(base_train_dataset, val_indices, eval_transform)\n",
BATCH_SIZE = 32
NUM_WORKERS = 2
NUM_EPOCHS_STAGE1 = 5
NUM_EPOCHS_STAGE2 = 3

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print('Train exists:', TRAIN_DIR.exists())
print('Val exists:', VAL_DIR.exists())
print('Test exists:', TEST_DIR.exists())

## 2. Подготовка данных

Аугментации применяются только к обучающей выборке, чтобы повысить устойчивость модели к поворотам, кадрированию и изменениям условий съёмки. Это важно для задач проверки подделок, потому что признаки фальсификации могут проявляться по-разному в зависимости от ракурса, качества печати и уровня сжатия.

Если внутри загруженного набора нет отдельной `val`-папки, ноутбук автоматически разобьёт train на train/val. Структура реального датасета может немного отличаться, поэтому ниже есть автоопределение split-директорий.

In [ ]:
class TransformedSubset(Dataset):
    def __init__(self, dataset, indices, transform):
        self.dataset = dataset
        self.indices = list(indices)
        self.transform = transform
        self.classes = dataset.classes

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        image, label = self.dataset[self.indices[index]]
        image = self.transform(image)
        return image, label


def find_split_dir(root: Path, candidates, required=True):
    candidate_names = {name.lower() for name in candidates}
    search_roots = [root] + [path for path in root.rglob('*') if path.is_dir()]

    for path in search_roots:
        if path.name.lower() in candidate_names:
            return path

    if required:
        raise FileNotFoundError(f'Не найдена папка из списка: {candidates} внутри {root}')
    return None


def build_datasets(train_dir: Path, val_dir: Path | None, test_dir: Path):
    base_train_dataset = datasets.ImageFolder(root=str(train_dir), transform=None)

    if val_dir is not None and val_dir.exists():
        train_dataset = datasets.ImageFolder(root=str(train_dir), transform=train_transform)
        val_dataset = datasets.ImageFolder(root=str(val_dir), transform=eval_transform)
    else:
        val_size = max(1, int(0.2 * len(base_train_dataset)))
        train_size = len(base_train_dataset) - val_size
        permutation = torch.randperm(len(base_train_dataset), generator=torch.Generator().manual_seed(42)).tolist()
        train_indices = permutation[:train_size]
        val_indices = permutation[train_size:]
        train_dataset = TransformedSubset(base_train_dataset, train_indices, train_transform)
        val_dataset = TransformedSubset(base_train_dataset, val_indices, eval_transform)

    test_dataset = datasets.ImageFolder(root=str(test_dir), transform=eval_transform)
    return train_dataset, val_dataset, test_dataset


TRAIN_DIR = find_split_dir(DATASET_ROOT, ['train', 'Train', 'training', 'Training'])
VAL_DIR = find_split_dir(DATASET_ROOT, ['val', 'Val', 'valid', 'Valid', 'validation', 'Validation'], required=False)
TEST_DIR = find_split_dir(DATASET_ROOT, ['test', 'Test', 'testing', 'Testing'])

print('Train dir:', TRAIN_DIR)
print('Val dir:', VAL_DIR)
print('Test dir:', TEST_DIR)

if TRAIN_DIR.exists() and TEST_DIR.exists():
    train_dataset, val_dataset, test_dataset = build_datasets(TRAIN_DIR, VAL_DIR if VAL_DIR is not None and VAL_DIR.exists() else None, TEST_DIR)
    class_names = train_dataset.dataset.classes if hasattr(train_dataset, 'dataset') else train_dataset.classes
    print('Classes:', class_names)
    print('Train size:', len(train_dataset))
    print('Val size:', len(val_dataset))
    print('Test size:', len(test_dataset))
else:
    train_dataset = val_dataset = test_dataset = None
    class_names = ['real', 'fake']
    print('Датасет не найден. Укажите корректные пути в DATASET_ROOT.')

In [ ]:
def denormalize(image_tensor: torch.Tensor) -> np.ndarray:
    image = image_tensor.detach().cpu().permute(1, 2, 0).numpy()
    image = image * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    return np.clip(image, 0, 1)

def show_batch(dataset, n=9):
    if dataset is None:
        return
    indices = np.random.choice(len(dataset), size=min(n, len(dataset)), replace=False)
    rows = int(np.ceil(len(indices) / 3))
    plt.figure(figsize=(12, 4 * rows))
    for i, idx in enumerate(indices, start=1):
        image, label = dataset[idx]
        plt.subplot(rows, 3, i)
        plt.imshow(denormalize(image))
        plt.title(class_names[label])
        plt.axis('off')
    plt.tight_layout()

show_batch(train_dataset)

In [ ]:
def make_loader(dataset, shuffle):
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

if train_dataset is not None:
    train_loader = make_loader(train_dataset, shuffle=True)
    val_loader = make_loader(val_dataset, shuffle=False)
    test_loader = make_loader(test_dataset, shuffle=False)
    print('Batches per epoch:', len(train_loader))
else:
    train_loader = val_loader = test_loader = None

## 3. Обоснование выбора архитектуры

В качестве базовой модели выбрана ResNet50. Причины выбора:
- остаточные связи позволяют глубокой сети устойчиво обучаться без деградации градиента;
- предобучение на ImageNet даёт полезные низкоуровневые и среднеуровневые признаки, которые хорошо переносятся на задачу анализа текстур и локальных артефактов;
- архитектура достаточно выразительна для извлечения мелких различий между подлинными и поддельными изображениями;
- ResNet50 обычно даёт хороший баланс между качеством и вычислительной стоимостью, особенно в режиме transfer learning.

На практике бинарная классификация подделок часто выигрывает от глубокой, но предобученной модели: она уже умеет выделять края, структуры и текстуры, а задача дообучения сводится к адаптации признаков к специфике домена.

In [ ]:
def build_model(num_classes: int = 1, freeze_backbone: bool = True):
    weights = models.ResNet50_Weights.DEFAULT
    model = models.resnet50(weights=weights)

    if freeze_backbone:
        for parameter in model.parameters():
            parameter.requires_grad = False

    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.5),
        nn.Linear(256, num_classes),
    )
    return model

model = build_model().to(device)
print(model.fc)

In [ ]:
def count_trainable_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

print('Trainable parameters:', count_trainable_parameters(model))

with torch.no_grad():
    dummy = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
    print('Logits shape:', model(dummy).shape)

## 4. Двухэтапное обучение

Этап 1: обучается только добавленная классификационная голова, а базовая сеть ResNet50 остаётся замороженной.

Этап 2: размораживается верхний блок `layer4`, и модель дообучается с очень маленьким learning rate. Это позволяет адаптировать высокоуровневые признаки к задаче обнаружения подделок без разрушения уже полезных представлений.

In [ ]:
criterion = nn.BCEWithLogitsLoss()

def move_batch_to_device(batch):
    images, targets = batch
    images = images.to(device, non_blocking=True)
    targets = targets.float().unsqueeze(1).to(device, non_blocking=True)
    return images, targets

def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    all_probs = []
    all_targets = []

    for batch in loader:
        images, targets = move_batch_to_device(batch)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = criterion(logits, targets)

        if is_train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * images.size(0)
        all_probs.append(torch.sigmoid(logits).detach().cpu())
        all_targets.append(targets.detach().cpu())

    probs = torch.cat(all_probs).squeeze(1).numpy()
    targets = torch.cat(all_targets).squeeze(1).numpy().astype(int)
    preds = (probs >= 0.5).astype(int)
    avg_loss = total_loss / len(loader.dataset)
    accuracy = (preds == targets).mean()
    return avg_loss, accuracy, preds, targets, probs

def fit_model(model, train_loader, val_loader, epochs, lr, weight_decay=1e-4):
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=weight_decay)
    history = {key: [] for key in ['train_loss', 'train_acc', 'val_loss', 'val_acc']}
    best_state = None
    best_val_acc = -1.0

    for epoch in range(epochs):
        train_loss, train_acc, _, _, _ = run_epoch(model, train_loader, optimizer)
        with torch.no_grad():
            val_loss, val_acc, _, _, _ = run_epoch(model, val_loader, optimizer=None)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f'Epoch {epoch + 1}/{epochs} | train loss={train_loss:.4f} acc={train_acc:.4f} | val loss={val_loss:.4f} acc={val_acc:.4f}')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history

def plot_history(history, title):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(epochs, history['train_loss'], label='train')
    axes[0].plot(epochs, history['val_loss'], label='val')
    axes[0].set_title(f'{title} | Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()

    axes[1].plot(epochs, history['train_acc'], label='train')
    axes[1].plot(epochs, history['val_acc'], label='val')
    axes[1].set_title(f'{title} | Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    plt.tight_layout()

In [ ]:
history_stage1 = None

if train_loader is not None:
    model_stage1 = build_model(freeze_backbone=True).to(device)
    model_stage1, history_stage1 = fit_model(
        model_stage1,
        train_loader,
        val_loader,
        epochs=NUM_EPOCHS_STAGE1,
        lr=1e-3,
    )
    plot_history(history_stage1, 'Stage 1')
else:
    model_stage1 = model
    print('Пропуск обучения: датасет не найден.')

In [ ]:
def unfreeze_top_layers(model: nn.Module, n_blocks: int = 1):
    for parameter in model.parameters():
        parameter.requires_grad = False

    for parameter in model.fc.parameters():
        parameter.requires_grad = True

    for parameter in model.layer4.parameters():
        parameter.requires_grad = True

    if n_blocks >= 2:
        for parameter in model.layer3.parameters():
            parameter.requires_grad = True

    return model

history_stage2 = None

if train_loader is not None:
    model_stage2 = copy.deepcopy(model_stage1)
    model_stage2 = unfreeze_top_layers(model_stage2, n_blocks=1).to(device)
    model_stage2, history_stage2 = fit_model(
        model_stage2,
        train_loader,
        val_loader,
        epochs=NUM_EPOCHS_STAGE2,
        lr=1e-5,
        weight_decay=1e-4,
    )
    plot_history(history_stage2, 'Stage 2')
else:
    model_stage2 = model_stage1

## 5. Оценка качества

Для интерпретации результатов используем accuracy, precision, recall, F1-score и confusion matrix. Для задач безопасности часто особенно важен recall класса "поддельный", потому что пропуск подделки обычно опаснее ложного срабатывания.

In [ ]:
def evaluate(model, loader):
    model.eval()
    all_probs = []
    all_targets = []
    total_loss = 0.0

    with torch.no_grad():
        for batch in loader:
            images, targets = move_batch_to_device(batch)
            logits = model(images)
            loss = criterion(logits, targets)
            total_loss += loss.item() * images.size(0)
            all_probs.append(torch.sigmoid(logits).cpu())
            all_targets.append(targets.cpu())

    probs = torch.cat(all_probs).squeeze(1).numpy()
    targets = torch.cat(all_targets).squeeze(1).numpy().astype(int)
    preds = (probs >= 0.5).astype(int)
    metrics = {
        'loss': total_loss / len(loader.dataset),
        'accuracy': (preds == targets).mean(),
        'precision': precision_score(targets, preds, zero_division=0),
        'recall': recall_score(targets, preds, zero_division=0),
        'f1': f1_score(targets, preds, zero_division=0),
    }
    return metrics, targets, preds, probs

def plot_confusion_matrix(targets, preds, labels):
    matrix = confusion_matrix(targets, preds)
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(matrix, cmap='Blues')
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title('Confusion Matrix')
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, matrix[i, j], ha='center', va='center', color='black')
    fig.colorbar(im, ax=ax)
    plt.tight_layout()

if test_loader is not None:
    metrics, targets, preds, probs = evaluate(model_stage2, test_loader)
    print(metrics)
    print(classification_report(targets, preds, target_names=class_names, zero_division=0))
    plot_confusion_matrix(targets, preds, class_names)
else:
    metrics = targets = preds = probs = None
    print('Датасет не найден, оценка не выполнена.')

## 6. Интерпретация ошибок

После обучения полезно изучить ошибки модели. Если поддельное изображение имеет высокое качество печати, мало артефактов или выглядит почти как оригинал, модель может ошибочно отнести его к классу "подлинное". Наоборот, дефекты освещения, сильное сжатие или нестандартный фон могут провоцировать ложные тревоги.

In [ ]:
def show_misclassified(dataset, preds, targets, max_items=9):
    if dataset is None or preds is None or targets is None:
        return

    errors = np.where(preds != targets)[0]
    if len(errors) == 0:
        print('Ошибок на выбранной выборке не найдено.')
        return

    n = min(max_items, len(errors))
    plt.figure(figsize=(12, 4 * int(np.ceil(n / 3))))
    for i, idx in enumerate(errors[:n], start=1):
        image, label = dataset[idx]
        plt.subplot(int(np.ceil(n / 3)), 3, i)
        plt.imshow(denormalize(image))
        plt.title(f'true={class_names[label]} | pred={class_names[preds[idx]]}')
        plt.axis('off')
    plt.tight_layout()

if test_loader is not None:
    show_misclassified(test_dataset, preds, targets)

## 7. Grad-CAM

Grad-CAM показывает, какие области изображения сильнее всего повлияли на решение модели. Для задачи поиска подделок это полезно, потому что можно проверить, опирается ли сеть на релевантные локальные артефакты, а не на случайный фон или посторонние признаки.

In [ ]:
def get_last_conv_layer(model: nn.Module):
    return model.layer4[-1]

def make_gradcam_heatmap(model: nn.Module, image_tensor: torch.Tensor, target_class: int | None = None):
    model.eval()
    activations = {}
    gradients = {}
    target_layer = get_last_conv_layer(model)

    def forward_hook(module, inputs, output):
        activations['value'] = output.detach()

    def backward_hook(module, grad_input, grad_output):
        gradients['value'] = grad_output[0].detach()

    forward_handle = target_layer.register_forward_hook(forward_hook)
    backward_handle = target_layer.register_full_backward_hook(backward_hook)

    image_tensor = image_tensor.unsqueeze(0).to(device)
    image_tensor.requires_grad_(True)
    logits = model(image_tensor)
    if target_class is None:
        target_class = int((torch.sigmoid(logits) >= 0.5).item())

    score = logits[:, 0]
    model.zero_grad(set_to_none=True)
    score.backward()

    forward_handle.remove()
    backward_handle.remove()

    acts = activations['value'][0]
    grads = gradients['value'][0]
    weights = grads.mean(dim=(1, 2), keepdim=True)
    cam = torch.relu((weights * acts).sum(dim=0))
    cam = cam - cam.min()
    cam = cam / (cam.max() + 1e-8)
    return cam.cpu().numpy()

def show_gradcam(model, dataset, index=0):
    if dataset is None:
        return
    image, label = dataset[index]
    heatmap = make_gradcam_heatmap(model, image)
    original = denormalize(image)

    plt.figure(figsize=(14, 5))
    plt.subplot(1, 3, 1)
    plt.imshow(original)
    plt.title(f'Original: {class_names[label]}')
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(heatmap, cmap='jet')
    plt.title('Grad-CAM heatmap')
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(original)
    plt.imshow(heatmap, cmap='jet', alpha=0.4)
    plt.title('Overlay')
    plt.axis('off')
    plt.tight_layout()

if test_loader is not None and len(test_dataset) > 0:
    show_gradcam(model_stage2, test_dataset, index=0)

## 8. Вывод

В рамках отчёта выбрана ResNet50 как компромисс между выразительностью и устойчивостью transfer learning. Сначала сеть используется как фиксированный извлекатель признаков, затем выполняется fine-tuning верхнего блока для адаптации к особенностям задачи.

При защите стоит отдельно подчеркнуть:
- почему именно глубина и остаточные связи полезны для распознавания мелких артефактов;
- почему предобучение снижает требования к объёму датасета;
- как интерпретируются precision, recall, F1 и confusion matrix;
- какие ошибки опаснее в контексте задачи безопасности.